# Sequential self-play training

Starting from the round-9, 20k-throw model, each iteration generates 500 states after seven throws, searches throws 8--10, and trains on the three labelled intermediate states per rollout.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from bot import ThrowPerturbations
from scratch import self_play_training as self_play

In [2]:
# Experiment parameters.  The defaults match sequential_training.ipynb.
seed = 0
num_iterations = 30
num_states_per_iteration = 500
grid_size = (10, 10, 4)
state_generator_version = 'turn_based'
halflife = 5.0
max_stones = 10
num_stones_per_side = 5
hidden_layer_size = 20
batch_size = 100
training_iterations = 30
learning_rate = 0.5
robust_top_fraction = 0.05
perturbation_angle_offsets_deg = (0.0, -0.2, 0.2)
perturbation_probabilities = (0.5, 0.25, 0.25)
resume_from = None  # Set to an integer checkpoint to continue an interrupted run.

data_root = Path('/home/vietafan/curling_data')
initial_run_dir = data_root / 'N5_rows20000_stages2' / 'sequential_experiment'
initial_model_path = initial_run_dir / 'm_9.npz'
initial_dataset_path = initial_run_dir / 'D_9.npz'
artifact_dir = data_root / 'self_play_N5_round9_20k'

config = self_play.SelfPlayConfig(
    num_states=num_states_per_iteration,
    num_iterations=num_iterations,
    grid_size=grid_size,
    state_generator_version=state_generator_version,
    halflife=halflife,
    max_stones=max_stones,
    num_stones_per_side=num_stones_per_side,
    hidden_layer_size=hidden_layer_size,
    batch_size=batch_size,
    training_iterations=training_iterations,
    learning_rate=learning_rate,
    robust_top_fraction=robust_top_fraction,
    perturbations=ThrowPerturbations(
        angle_offsets_deg=perturbation_angle_offsets_deg,
        probabilities=perturbation_probabilities,
    ),
)
print(f'retention probability: {config.retention_probability():.4f}')

retention probability: 0.8706


In [4]:
results = self_play.run_self_play(
    config=config,
    artifact_dir=artifact_dir,
    initial_model_path=initial_model_path,
    initial_dataset_path=initial_dataset_path,
    seed=seed,
    resume_from=resume_from,
)
results

{1: {'new_rows': 1500,
  'training_rows': 18939,
  'loss': [0.6238303407633381,
   0.643917202643899,
   0.622379426005555,
   0.6235806492573817,
   0.6256653774611193,
   0.6380289941411472,
   0.6276770110607801,
   0.618918588933575,
   0.6064891494503174,
   0.6126436910601065,
   0.625053545048508,
   0.6026713915210347,
   0.6073785885408294,
   0.6055509153252138,
   0.6074523185708629,
   0.6017819288537396,
   0.5942224676684689,
   0.5976315636891492,
   0.5941980135784294,
   0.5898770513032859,
   0.58918548923594,
   0.5864393411697373,
   0.5846240242990133,
   0.5839246973248188,
   0.5831602694997918,
   0.582230235967128,
   0.5824168828018306,
   0.5816758171076009,
   0.5815736410575569,
   0.5814864693028534]},
 2: {'new_rows': 1500,
  'training_rows': 17994,
  'loss': [0.5627421529191445,
   0.5581682445736117,
   0.559120968556919,
   0.5528830239137978,
   0.5568658030093853,
   0.5847595647785693,
   0.5557503635064523,
   0.5536411820790451,
   0.5491542632058

In [5]:
# Each iteration writes D_i.npz, X_i.npz, and m_i.npz.
for i, record in results.items():
    print(i, record['new_rows'], record['training_rows'], record['loss'][-1])

1 1500 18939 0.5814864693028534
2 1500 17994 0.5207986290517325
3 1500 17091 0.5178852256332904
4 1500 16353 0.5583353265666702
5 1500 15762 0.5543215810702677
6 1500 15209 0.5478280897163202
7 1500 14673 0.5161297628210813
8 1500 14262 0.45593144678824277
9 1500 13894 0.5259429830662579
10 1500 13687 0.5112843023173226
11 1500 13419 0.4985658266006223
12 1500 13215 0.43770487321618706
13 1500 12999 0.5089056262821303
14 1500 12924 0.5703044628303825
15 1500 12762 0.5570660927772038
16 1500 12593 0.5303983952096895
17 1500 12592 0.5236948141829764
18 1500 12518 0.5539712987309423
19 1500 12419 0.5831432977668279
20 1500 12278 0.5630568361073114
21 1500 12183 0.4860746585036637
22 1500 12128 0.5213092179658354
23 1500 12074 0.5187996222238744
24 1500 12019 0.5532309938359077
25 1500 11940 0.5671458164796969
26 1500 11929 0.6118201208144526
27 1500 11849 0.5224090981082151
28 1500 11814 0.5994668348698542
29 1500 11809 0.5910784263876285
30 1500 11748 0.6122363134696406
